[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeiraDavidson/math_module2/blob/main/notebooks/ch5_exponentials.ipynb)

# Chapter 5 — Companion Notebook
## The Function That Is Its Own Slope
*Why one strange number between 2 and 3 refuses to change*

Three live experiments on the number $e$. You will drag a base $a$ and watch the curve $y=a^x$ tilt away from the $y$-axis until its crossing slope $M(a)=\ln a$ hits exactly $1$ — that base *is* $e$. Then you will see what "its own derivative" looks like, where the steepness of $y=e^x$ equals its height at every point. Finally you will run a decay simulator and watch a quantity halve, then halve again, at evenly spaced half-lives.

> **How to use this notebook.** Run every cell from the top (Shift+Enter). Each widget below is live — drag the sliders and watch the picture respond. Requires `ipywidgets` and `matplotlib`.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from ipywidgets import (interact, interactive, fixed, Dropdown, Checkbox,
                        Button, Output, VBox, HBox,
                        FloatSlider, IntSlider, FloatLogSlider)
from IPython.display import HTML, display, Markdown

# Book 2 palette — matches the printed figures in figures/png/
BLUE="#1f4e79"; RED="#c0392b"; GREEN="#2e8b57"; ORANGE="#e08e0b"
PURPLE="#6c3483"; TEAL="#1b9e9e"; GRAY="#7f8c8d"; DARK="#2c3e50"
SHADE="#9fc5e8"; SHADE2="#f4b6ad"; SHADEG="#a9dfbf"; LIGHT="#d6dbdf"

plt.rcParams.update({
    "figure.figsize": (7.2, 4.5), "figure.dpi": 96,
    "font.family": "serif", "mathtext.fontset": "cm", "font.size": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": LIGHT, "grid.linewidth": 0.7,
    "lines.linewidth": 2.2, "lines.solid_capstyle": "round",
})

def axes(ax=None, title=None):
    '''A clean axes with a light grid; returns the axes.'''
    if ax is None:
        _, ax = plt.subplots()
    if title:
        ax.set_title(title, color=DARK)
    return ax


## 1. Hunting for the base that leaves the axis at slope 1

Take any exponential $y=a^x$ and differentiate it from scratch. The $a^x$ factors straight out of the difference quotient and *never changes as $h$ shrinks*, leaving one lonely number behind:

$$\frac{d}{dx}\,a^{x} = M(a)\,a^{x},\qquad M(a)=\lim_{h\to 0}\frac{a^{h}-1}{h}.$$

Every exponential is *already* proportional to its own slope — off only by that constant $M(a)$. And $M(a)$ has a picture: plug in $x=0$ and, since $a^0=1$, the slope of $y=a^x$ right where it **crosses the $y$-axis** is exactly $M(a)$.

So our whole quest reduces to one question: *for which base does the curve leave the axis at slope exactly $1$?* Drag $a$ below. The blue curve is $y=a^x$; the red line is its tangent at the crossing point $(0,1)$, and its slope is the live number $M(a)=\ln a$. Push $a$ until that slope reads $1$ — you will be sitting on $a=e\approx 2.718$, the base whose curve refuses to leave the axis at any slope but one.

In [2]:
E = np.e  # 2.71828..., the base we are hunting for

def _base_slope(a=2.0, show_family=True):
    # M(a) = ln a is the slope of a^x at x=0 (the y-axis crossing).
    M = np.log(a)
    x = np.linspace(-2.2, 2.2, 400)

    fig, ax = plt.subplots()
    axes(ax)

    # Faint family of other bases, for context.
    if show_family:
        for b in (1.5, 2.0, E, 3.0, 4.0):
            ax.plot(x, b ** x, color=GRAY, lw=1.1, alpha=0.30, zorder=1)

    # The chosen curve y = a^x.
    ax.plot(x, a ** x, color=BLUE, lw=2.6, zorder=3,
            label=f'$y = a^x$,  $a={a:.3f}$')

    # Tangent at the y-axis crossing (0, 1): slope = M(a) = ln a.
    xt = np.linspace(-1.4, 1.4, 2)
    ax.plot(xt, 1.0 + M * xt, color=RED, lw=2.2, ls='--', zorder=4,
            label=f'tangent at $(0,1)$, slope $= \\ln a = {M:.3f}$')
    ax.scatter([0], [1], s=46, color=RED, zorder=5)

    # Live verdict on whether we have found e.
    if abs(M - 1.0) < 0.01:
        verdict = (f'$M(a) = {M:.3f} \\approx 1$  \u2014  '
                   f'this is $e \\approx {E:.3f}$!')
        vcolor = GREEN
    elif M < 1.0:
        verdict = f'$M(a) = {M:.3f} < 1$  \u2014  too gentle, push $a$ up'
        vcolor = DARK
    else:
        verdict = f'$M(a) = {M:.3f} > 1$  \u2014  too steep, ease $a$ down'
        vcolor = DARK
    ax.set_title(verdict, color=vcolor, fontsize=13)

    ax.axhline(0, color=DARK, lw=0.8)
    ax.axvline(0, color=DARK, lw=0.8)
    ax.set_xlim(-2.2, 2.2)
    ax.set_ylim(-0.5, 8)
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    ax.legend(loc='upper left', framealpha=0.92)
    plt.show()

interact(_base_slope,
         a=FloatSlider(value=2.0, min=1.2, max=4.0, step=0.01,
                       description='base $a$', readout_format='.2f',
                       continuous_update=False),
         show_family=Checkbox(value=True, description='show other bases'));

interactive(children=(FloatSlider(value=2.0, continuous_update=False, description='base $a$', max=4.0, min=1.2…

## 2. "Its own derivative" — slope equals height

Setting $M(e)=1$ collapses the boxed rule to the headline of the whole chapter:

$$\frac{d}{dx}\,e^{x} = e^{x}.$$

That equation is a *geometric promise*: at any point $x_0$ the tangent to $y=e^x$ has slope $e^{x_0}$, which is the very height of the curve there. Where the curve sits at height $5$, it climbs at slope $5$. Slide $x_0$ along the curve and watch the green height and the red slope stay locked to the same number — the shaded triangle has rise equal to its run scaled by exactly that value.

In [3]:
def _slope_equals_height(x0=0.6):
    x = np.linspace(-2.5, 2.0, 400)
    y = np.exp(x)
    h = np.exp(x0)          # height = slope, the whole point

    fig, ax = plt.subplots()
    axes(ax, '$y=e^x$:  slope at $x_0$ equals the height $e^{x_0}$')
    ax.plot(x, y, color=BLUE, lw=2.6, zorder=3, label='$y = e^x$')

    # Tangent line through (x0, h) with slope h.
    xt = np.linspace(x0 - 1.3, x0 + 1.3, 2)
    ax.plot(xt, h + h * (xt - x0), color=RED, lw=2.2, ls='--',
            zorder=4, label=f'tangent, slope $= {h:.3f}$')

    # Drop line showing the height, and a rise/run slope triangle.
    ax.vlines(x0, 0, h, color=GREEN, lw=2.0, zorder=2)
    ax.plot([x0, x0 + 1], [h, h], color=GRAY, lw=1.4, zorder=2)
    ax.plot([x0 + 1, x0 + 1], [h, h + h], color=GRAY, lw=1.4, zorder=2)
    ax.fill([x0, x0 + 1, x0 + 1], [h, h, h + h],
            color=SHADE2, alpha=0.55, zorder=1)
    ax.annotate(f'run $=1$', xy=(x0 + 0.5, h), xytext=(x0 + 0.5, h - 0.9),
                ha='center', color=GRAY, fontsize=11)
    ax.annotate(f'rise $= {h:.2f}$', xy=(x0 + 1, h + h / 2),
                xytext=(x0 + 1.1, h + h / 2), va='center',
                color=DARK, fontsize=11)

    ax.scatter([x0], [h], s=46, color=RED, zorder=5)
    ax.annotate(f'height $= e^{{{x0:.2f}}} = {h:.3f}$',
                xy=(x0, h), xytext=(x0 - 2.3, h + 0.6),
                color=GREEN, fontsize=12,
                arrowprops=dict(arrowstyle='->', color=GREEN, lw=1.4))

    ax.axhline(0, color=DARK, lw=0.8)
    ax.axvline(0, color=DARK, lw=0.8)
    ax.set_xlim(-2.5, 2.0)
    ax.set_ylim(-0.6, 8)
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    ax.legend(loc='upper left', framealpha=0.92)
    plt.show()

interact(_slope_equals_height,
         x0=FloatSlider(value=0.6, min=-2.0, max=1.9, step=0.01,
                        description='point $x_0$', readout_format='.2f',
                        continuous_update=False));

interactive(children=(FloatSlider(value=0.6, continuous_update=False, description='point $x_0$', max=1.9, min=…

## 3. Decay and half-life

Flip the sign of the rate constant and the same function *decays*. Written in terms of its **half-life** $T$ — the time for the amount to fall to half of whatever it was — the model is

$$N(t) = N_{0}\left(\tfrac12\right)^{t/T} = N_0\,e^{-\lambda t},\qquad \lambda = \frac{\ln 2}{T}.$$

The defining feature of exponential decay is that this halving is *memoryless*: it takes one half-life $T$ to go from $N_0$ to $N_0/2$, another $T$ to reach $N_0/4$, another to reach $N_0/8$ — no matter where you start counting. Drag the half-life $T$ (and the starting amount $N_0$). The dashed guides mark $t=T,2T,3T,4T$; at each one the quantity is exactly halved from the step before, and the labels read off the value.

In [ ]:
def _decay(T=2.0, N0=400.0, n_halves=4):
    t = np.linspace(0, n_halves * T * 1.05, 400)
    N = N0 * 0.5 ** (t / T)

    fig, ax = plt.subplots()
    lam = np.log(2) / T
    axes(ax, f'$N(t)=N_0(1/2)^{{t/T}}$,  half-life $T={T:.2f}$,  '
             f'$\\lambda=\\ln 2/T={lam:.3f}$')
    ax.fill_between(t, 0, N, color=SHADE, alpha=0.35, zorder=0)
    ax.plot(t, N, color=BLUE, lw=2.6, zorder=3, label='$N(t)$')

    # Mark successive half-lives t = T, 2T, 3T, ... with guides.
    for k in range(0, n_halves + 1):
        tk = k * T
        Nk = N0 * 0.5 ** k     # = N0 / 2^k
        # vertical guide up to the curve, horizontal guide to the axis
        ax.vlines(tk, 0, Nk, color=RED, lw=1.3, ls='--', zorder=2)
        ax.hlines(Nk, 0, tk, color=GREEN, lw=1.3, ls=':', zorder=2)
        ax.scatter([tk], [Nk], s=40, color=RED, zorder=4)
        label = f'{Nk:.4g}' if k else f'$N_0={Nk:.4g}$'
        ax.annotate(label, xy=(tk, Nk), xytext=(tk + 0.12 * T, Nk + 0.04 * N0),
                    color=DARK, fontsize=11)
        if k:
            ax.annotate(f'${k}T$', xy=(tk, 0), xytext=(tk, -0.07 * N0),
                        ha='center', color=RED, fontsize=11)

    ax.set_xlim(0, n_halves * T * 1.05)
    ax.set_ylim(-0.1 * N0, 1.12 * N0)
    ax.set_xlabel('time $t$'); ax.set_ylabel('amount $N$')
    ax.legend(loc='upper right', framealpha=0.92)
    plt.show()

interact(_decay,
         T=FloatSlider(value=2.0, min=0.5, max=5.0, step=0.1,
                       description='half-life $T$', readout_format='.1f',
                       continuous_update=False),
         N0=FloatSlider(value=400.0, min=50.0, max=1000.0, step=50.0,
                        description='start $N_0$', readout_format='.0f',
                        continuous_update=False),
         n_halves=IntSlider(value=4, min=2, max=6, step=1,
                            description='# half-lives'));

---

*Three pictures, one idea.* The first widget shows that picking the base is the same as picking a crossing slope $M(a)=\ln a$, and $e$ is the unique base that picks slope $1$. The second shows the payoff — at $a=e$ the curve's height *is* its slope everywhere. The third turns the same exponential, run with a negative rate, into the half-life arithmetic you can now derive rather than memorize: $t_{1/2} = \ln 2/\lambda$.